# Autoresearch v2 — Enhanced Edition

Port of [Karpathy's autoresearch](https://github.com/karpathy/autoresearch) for Colab Pro, enhanced with ideas from [AlphaEvolve](https://deepmind.google/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/), [AIDE](https://arxiv.org/html/2502.13138v1), [pi-autoresearch](https://github.com/davebcn87/pi-autoresearch), and [The AI Scientist](https://sakana.ai/ai-scientist/).

### What's new in v2
| Feature | Inspired by | What it does |
|---------|-------------|-------------|
| Live Dashboard | pi-autoresearch | Real-time HTML dashboard updates after each experiment |
| Dual-Model Strategy | AlphaEvolve | Haiku brainstorms ideas cheaply, Sonnet implements the best |
| Syntax Pre-Validation | AI Scientist | Catches broken code *before* wasting 5 min training |
| Tree Search + Lineage | AIDE | Tracks parent-child relationships between experiments |
| Experiment Taxonomy | AI Scientist | Auto-categorizes experiments, shows which types work best |
| Auto-Resume | pi-autoresearch | Survives Colab disconnects — re-run cell to continue |
| HTML Summary Report | AI Scientist | Beautiful downloadable report when done |

---
### Before you begin
1. **Runtime > Change runtime type > GPU** (T4 fine, A100 better)
2. You need an **Anthropic API key** (Cell 5)
3. Colab Pro recommended for long runs

---
## Cell 1: Environment Setup

In [ ]:
#@title Cell 1: Install Dependencies & Detect GPU

import subprocess, sys, os

packages = [
    "rustbpe>=0.1.0",
    "tiktoken>=0.11.0",
    "pyarrow>=21.0.0",
    "anthropic>=0.49.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
cap = torch.cuda.get_device_capability()

if gpu_mem_gb >= 70:   GPU_TIER = "H100"
elif gpu_mem_gb >= 35:  GPU_TIER = "A100"
elif gpu_mem_gb >= 20:  GPU_TIER = "L4"
else:                   GPU_TIER = "T4"

print(f"GPU: {gpu_name} | VRAM: {gpu_mem_gb:.1f} GB | Tier: {GPU_TIER}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print("Environment ready!")

---
## Cell 2: Clone & Prepare Data

In [ ]:
#@title Cell 2: Clone Autoresearch & Download Data
num_shards = 4  #@param {type:"integer"}

import os
WORK_DIR = "/content/autoresearch"

if not os.path.exists(WORK_DIR):
    !git clone https://github.com/karpathy/autoresearch.git {WORK_DIR}
else:
    print("Repo already cloned.")

os.chdir(WORK_DIR)
!python prepare.py --num-shards {num_shards}

cache_dir = os.path.expanduser("~/.cache/autoresearch")
n_shards = len([f for f in os.listdir(os.path.join(cache_dir, "data")) if f.endswith('.parquet')])
print(f"\nData shards: {n_shards} | Tokenizer: ready")

---
## Cell 3: Patch train.py for Colab GPU

In [ ]:
#@title Cell 3: Patch train.py for Colab Compatibility
#@markdown Replaces Flash Attention 3 with PyTorch SDPA, scales model for your GPU.

import os, re
os.chdir("/content/autoresearch")

with open("train.py", "r") as f:
    original = f.read()
with open("train_original.py", "w") as f:
    f.write(original)

GPU_CONFIGS = {
    "H100": dict(DEPTH=8,  DEVICE_BATCH_SIZE=128, TOTAL_BATCH_SIZE=2**19, WINDOW_PATTERN='"SSSL"'),
    "A100": dict(DEPTH=8,  DEVICE_BATCH_SIZE=32,  TOTAL_BATCH_SIZE=2**17, WINDOW_PATTERN='"SSSL"'),
    "L4":   dict(DEPTH=6,  DEVICE_BATCH_SIZE=16,  TOTAL_BATCH_SIZE=2**16, WINDOW_PATTERN='"SL"'),
    "T4":   dict(DEPTH=4,  DEVICE_BATCH_SIZE=8,   TOTAL_BATCH_SIZE=2**15, WINDOW_PATTERN='"L"'),
}
cfg = GPU_CONFIGS[GPU_TIER]
patched = original

# Replace FA3 imports
fa3_block = """from kernels import get_kernel
cap = torch.cuda.get_device_capability()
# varunneal's FA3 is Hopper only, use kernels-community on non-Hopper GPUs
repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"
fa3 = get_kernel(repo).flash_attn_interface"""
patched = patched.replace(fa3_block, "# Colab: using PyTorch SDPA instead of FA3")

# Replace FA3 attention call with SDPA
old_attn = """        y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)
        y = y.contiguous().view(B, T, -1)"""
new_attn = """        q_sdpa = q.transpose(1, 2)
        k_sdpa = k.transpose(1, 2)
        v_sdpa = v.transpose(1, 2)
        if self.n_kv_head < self.n_head:
            rep = self.n_head // self.n_kv_head
            k_sdpa = k_sdpa.repeat_interleave(rep, dim=1)
            v_sdpa = v_sdpa.repeat_interleave(rep, dim=1)
        y = F.scaled_dot_product_attention(q_sdpa, k_sdpa, v_sdpa, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, -1)"""
patched = patched.replace(old_attn, new_attn)

# Scale hyperparams
patched = re.sub(r'^DEPTH = \d+', f'DEPTH = {cfg["DEPTH"]}', patched, flags=re.MULTILINE)
patched = re.sub(r'^DEVICE_BATCH_SIZE = \d+', f'DEVICE_BATCH_SIZE = {cfg["DEVICE_BATCH_SIZE"]}', patched, flags=re.MULTILINE)
patched = re.sub(r'^TOTAL_BATCH_SIZE = .*$', f'TOTAL_BATCH_SIZE = {cfg["TOTAL_BATCH_SIZE"]}', patched, flags=re.MULTILINE)
patched = re.sub(r'^WINDOW_PATTERN = .*$', f'WINDOW_PATTERN = {cfg["WINDOW_PATTERN"]}', patched, flags=re.MULTILINE)

flops_map = {"H100": "989.5e12", "A100": "312e12", "L4": "121e12", "T4": "65e12"}
patched = patched.replace("H100_BF16_PEAK_FLOPS = 989.5e12", f"GPU_PEAK_FLOPS = {flops_map[GPU_TIER]}")
patched = patched.replace("H100_BF16_PEAK_FLOPS", "GPU_PEAK_FLOPS")

if GPU_TIER == "T4":
    patched = patched.replace('dtype=torch.bfloat16', 'dtype=torch.float16')
    patched = patched.replace('.bfloat16()', '.half()')

with open("train.py", "w") as f:
    f.write(patched)

print(f"Patched for {GPU_TIER}: DEPTH={cfg['DEPTH']}, BATCH={cfg['DEVICE_BATCH_SIZE']}, TOTAL={cfg['TOTAL_BATCH_SIZE']}")
print(f"Attention: SDPA | Precision: {'fp16' if GPU_TIER=='T4' else 'bf16'}")

---
## Cell 4: Baseline Run

In [ ]:
#@title Cell 4: Run Baseline (~5-7 min)

import subprocess, os, json, time
os.chdir("/content/autoresearch")

print("Running baseline (~5-7 min with compilation)...")
result = subprocess.run(["python", "train.py"], capture_output=True, text=True, timeout=900)

with open("run.log", "w") as f:
    f.write(result.stdout + "\n" + result.stderr)

if result.returncode != 0:
    print("BASELINE FAILED!")
    for line in (result.stdout + result.stderr).split("\n")[-30:]:
        print(line)
else:
    val_bpb = peak_vram = None
    for line in result.stdout.split("\n"):
        if line.startswith("val_bpb:"):      val_bpb = float(line.split()[-1])
        if line.startswith("peak_vram_mb:"): peak_vram = float(line.split()[-1])

    if val_bpb:
        mem_gb = round(peak_vram / 1024, 1)

        # Initialize checkpoint.json (auto-resume state)
        checkpoint = {
            "gpu_tier": GPU_TIER,
            "baseline_bpb": val_bpb,
            "best_bpb": val_bpb,
            "best_train_py": open("train.py").read(),
            "experiments_completed": 0,
            "experiments": [{
                "id": "exp000",
                "parent_id": None,
                "val_bpb": val_bpb,
                "memory_gb": mem_gb,
                "status": "keep",
                "category": "baseline",
                "description": "baseline",
                "timestamp": time.time(),
                "elapsed_sec": 0
            }]
        }
        with open("checkpoint.json", "w") as f:
            json.dump(checkpoint, f, indent=2)

        print(f"\nBASELINE val_bpb: {val_bpb:.6f} | VRAM: {peak_vram:.0f} MB ({mem_gb} GB)")
        print("Checkpoint saved. Ready for experiments!")
    else:
        print("Could not parse val_bpb. Check run.log.")

---
## Cell 5: API Key

In [ ]:
#@title Cell 5: Configure Anthropic API Key
ANTHROPIC_API_KEY = ""  #@param {type:"string"}

import os
try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY')
    if key: ANTHROPIC_API_KEY = key; print("Loaded from Colab Secrets.")
except: pass

if not ANTHROPIC_API_KEY:
    raise ValueError("Set ANTHROPIC_API_KEY above or in Colab Secrets!")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
print(f"API key set (...{ANTHROPIC_API_KEY[-4:]})")

---
## Cell 6: Enhanced Autonomous Experiment Loop

The main event. Features:
- **Dual-model strategy** (Haiku brainstorms, Sonnet implements)
- **Syntax pre-validation** before training
- **Tree search lineage** tracking
- **Experiment taxonomy** with category tracking
- **Auto-resume** from checkpoint on disconnect
- **Live dashboard** updates after every experiment

In [ ]:
#@title Cell 6: Run Enhanced Experiment Loop
#@markdown **max_experiments**: Number of experiments (12 ~= 1 hour)
#@markdown
#@markdown **implementation_model**: Model that writes code
#@markdown
#@markdown **use_dual_model**: Use Haiku to brainstorm + Sonnet to implement (saves cost, explores more)

max_experiments = 12  #@param {type:"integer"}
implementation_model = "claude-sonnet-4-6"  #@param ["claude-sonnet-4-6", "claude-haiku-4-5-20251001", "claude-opus-4-6"]
use_dual_model = True  #@param {type:"boolean"}
brainstorm_model = "claude-haiku-4-5-20251001"

import anthropic
import subprocess
import json
import re
import os
import sys
import time
import py_compile
import tempfile
from datetime import datetime
from IPython.display import display, HTML, clear_output

os.chdir("/content/autoresearch")
client = anthropic.Anthropic()

# ─── Experiment Categories ──────────────────────────────────────────
CATEGORIES = [
    "architecture",   # model structure, layers, attention changes
    "optimizer",      # optimizer type, momentum, weight decay
    "hyperparameter", # learning rates, batch size, warmup
    "regularization", # dropout, normalization, weight init
    "activation",     # activation functions, gating
    "embedding",      # embeddings, positional encoding
    "scheduling",     # LR schedule, warmup/cooldown ratios
    "other"           # anything else
]

# ─── Utility Functions ──────────────────────────────────────────────

def read_file(path):
    with open(path, "r") as f: return f.read()

def write_file(path, content):
    with open(path, "w") as f: f.write(content)

def load_checkpoint():
    with open("checkpoint.json", "r") as f: return json.load(f)

def save_checkpoint(ckpt):
    with open("checkpoint.json", "w") as f: json.dump(ckpt, f, indent=2)

def validate_syntax(code):
    """Pre-validate Python syntax before burning 5 min on training."""
    tmp = tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w")
    tmp.write(code)
    tmp.close()
    try:
        py_compile.compile(tmp.name, doraise=True)
        return True, ""
    except py_compile.PyCompileError as e:
        return False, str(e)
    finally:
        os.unlink(tmp.name)

def run_training(timeout=600):
    """Run train.py, return (val_bpb, peak_vram_mb, success, error)."""
    try:
        result = subprocess.run(
            ["python", "train.py"],
            capture_output=True, text=True, timeout=timeout
        )
        log = result.stdout + "\n" + result.stderr
        write_file("run.log", log)
        if result.returncode != 0:
            return None, None, False, log[-2000:]
        val_bpb = peak_vram = None
        for line in log.split("\n"):
            if line.startswith("val_bpb:"):      val_bpb = float(line.split()[-1])
            if line.startswith("peak_vram_mb:"): peak_vram = float(line.split()[-1])
        if val_bpb is None:
            return None, None, False, log[-2000:]
        return val_bpb, peak_vram, True, ""
    except subprocess.TimeoutExpired:
        return None, None, False, "TIMEOUT: exceeded 10 minutes"
    except Exception as e:
        return None, None, False, str(e)

# ─── Live Dashboard ─────────────────────────────────────────────────

def render_dashboard(ckpt, status_msg=""):
    """Render a live HTML dashboard in the Colab output."""
    exps = ckpt["experiments"]
    n_total = len(exps) - 1  # exclude baseline
    n_keep = sum(1 for e in exps if e["status"] == "keep") - 1
    n_discard = sum(1 for e in exps if e["status"] == "discard")
    n_crash = sum(1 for e in exps if e["status"] == "crash")
    n_syntax = sum(1 for e in exps if e["status"] == "syntax_fail")
    baseline = ckpt["baseline_bpb"]
    best = ckpt["best_bpb"]
    improvement = baseline - best
    pct = (improvement / baseline * 100) if baseline > 0 else 0
    keep_rate = (n_keep / n_total * 100) if n_total > 0 else 0

    # Category stats
    cat_stats = {}
    for e in exps:
        c = e.get("category", "other")
        if c == "baseline": continue
        if c not in cat_stats: cat_stats[c] = {"total": 0, "kept": 0}
        cat_stats[c]["total"] += 1
        if e["status"] == "keep": cat_stats[c]["kept"] += 1

    # Chart data
    valid_exps = [e for e in exps if e["status"] in ("keep", "discard")]
    chart_labels = json.dumps([e["id"] for e in valid_exps])
    chart_bpbs = json.dumps([e["val_bpb"] for e in valid_exps])
    chart_colors = json.dumps(["#2ecc71" if e["status"]=="keep" else "#ddd" for e in valid_exps])
    chart_borders = json.dumps(["#27ae60" if e["status"]=="keep" else "#bbb" for e in valid_exps])

    # Running best line
    running_best = []
    current_best = baseline
    for e in valid_exps:
        if e["status"] == "keep" and e["val_bpb"] < current_best:
            current_best = e["val_bpb"]
        running_best.append(current_best)
    chart_running = json.dumps(running_best)

    # Lineage tree (kept experiments path)
    kept_chain = [e for e in exps if e["status"] == "keep"]
    lineage_html = ""
    for i, e in enumerate(kept_chain):
        arrow = "" if i == 0 else '<span style="color:#27ae60;margin:0 6px">&#8594;</span>'
        lineage_html += f'{arrow}<span style="background:#e8f5e9;border-radius:4px;padding:2px 8px;font-size:12px;border:1px solid #c8e6c9"><b>{e["id"]}</b> {e["val_bpb"]:.4f}</span>'

    # Category heatmap
    cat_html = ""
    for cat in sorted(cat_stats.keys()):
        s = cat_stats[cat]
        rate = s["kept"] / s["total"] * 100 if s["total"] > 0 else 0
        if rate >= 50: bg = "#c8e6c9"
        elif rate >= 20: bg = "#fff9c4"
        elif s["total"] > 0: bg = "#ffcdd2"
        else: bg = "#f5f5f5"
        cat_html += f'<div style="display:inline-block;margin:3px;padding:4px 10px;border-radius:6px;background:{bg};font-size:12px"><b>{cat}</b> {s["kept"]}/{s["total"]} ({rate:.0f}%)</div>'

    # Recent experiments table
    recent = exps[-8:] if len(exps) > 8 else exps
    rows_html = ""
    for e in reversed(recent):
        status_icon = {"keep": "<span style='color:#27ae60'>&#10004;</span>",
                       "discard": "<span style='color:#999'>&#10008;</span>",
                       "crash": "<span style='color:#e74c3c'>&#9888;</span>",
                       "syntax_fail": "<span style='color:#f39c12'>&#9998;</span>"}
        icon = status_icon.get(e["status"], "")
        bpb_str = f"{e['val_bpb']:.6f}" if e["val_bpb"] > 0 else "-"
        cat_badge = f'<span style="background:#e3f2fd;border-radius:3px;padding:1px 5px;font-size:10px">{e.get("category","")}</span>'
        parent = e.get("parent_id", "-") or "-"
        rows_html += f'<tr><td>{e["id"]}</td><td>{icon} {e["status"]}</td><td style="font-family:monospace">{bpb_str}</td><td>{cat_badge}</td><td style="font-size:11px;color:#555">{parent}</td><td style="font-size:12px">{e["description"][:50]}</td></tr>'

    html = f"""
    <style>
      .ar-dash {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 960px; }}
      .ar-cards {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 16px 0; }}
      .ar-card {{ background: #f8f9fa; border-radius: 10px; padding: 16px; text-align: center; border: 1px solid #e9ecef; }}
      .ar-card .val {{ font-size: 28px; font-weight: 700; line-height: 1.2; }}
      .ar-card .lbl {{ font-size: 11px; color: #888; text-transform: uppercase; letter-spacing: 0.5px; margin-top: 4px; }}
      .ar-table {{ width: 100%; border-collapse: collapse; font-size: 13px; margin-top: 12px; }}
      .ar-table th {{ background: #f1f3f5; padding: 8px; text-align: left; font-size: 11px; text-transform: uppercase; color: #666; }}
      .ar-table td {{ padding: 6px 8px; border-bottom: 1px solid #f1f3f5; }}
      .ar-section {{ margin: 18px 0 8px; font-size: 13px; font-weight: 600; color: #495057; text-transform: uppercase; letter-spacing: 0.5px; }}
      .ar-status {{ background: #e8f5e9; color: #2e7d32; padding: 8px 14px; border-radius: 8px; font-size: 13px; margin: 12px 0; }}
    </style>
    <div class="ar-dash">
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:4px">
        <span style="font-size:22px;font-weight:700">Autoresearch v2</span>
        <span style="background:#e3f2fd;color:#1565c0;padding:3px 10px;border-radius:12px;font-size:11px;font-weight:600">{ckpt.get('gpu_tier','?')}</span>
        <span style="color:#999;font-size:12px;margin-left:auto">{datetime.now().strftime('%H:%M:%S')}</span>
      </div>

      {'<div class="ar-status">' + status_msg + '</div>' if status_msg else ''}

      <div class="ar-cards">
        <div class="ar-card"><div class="val" style="color:#2e7d32">{best:.4f}</div><div class="lbl">Best BPB</div></div>
        <div class="ar-card"><div class="val">{improvement:.4f}</div><div class="lbl">Improvement</div></div>
        <div class="ar-card"><div class="val">{n_total}</div><div class="lbl">Experiments</div></div>
        <div class="ar-card"><div class="val" style="color:{'#2e7d32' if keep_rate > 20 else '#e65100'}">{keep_rate:.0f}%</div><div class="lbl">Keep Rate</div></div>
      </div>

      <div class="ar-cards" style="grid-template-columns: repeat(4, 1fr); gap: 8px;">
        <div style="text-align:center;font-size:12px"><span style="color:#2e7d32;font-weight:700;font-size:18px">{n_keep}</span><br><span style="color:#888">kept</span></div>
        <div style="text-align:center;font-size:12px"><span style="color:#999;font-weight:700;font-size:18px">{n_discard}</span><br><span style="color:#888">discarded</span></div>
        <div style="text-align:center;font-size:12px"><span style="color:#e74c3c;font-weight:700;font-size:18px">{n_crash}</span><br><span style="color:#888">crashed</span></div>
        <div style="text-align:center;font-size:12px"><span style="color:#f39c12;font-weight:700;font-size:18px">{n_syntax}</span><br><span style="color:#888">syntax fail</span></div>
      </div>

      <div class="ar-section">Progress</div>
      <canvas id="arChart" height="100"></canvas>

      <div class="ar-section">Lineage (kept chain)</div>
      <div style="overflow-x:auto;padding:8px 0">{lineage_html}</div>

      <div class="ar-section">Category Success Rates</div>
      <div style="padding:4px 0">{cat_html if cat_html else '<span style="color:#999;font-size:12px">No experiments yet</span>'}</div>

      <div class="ar-section">Recent Experiments</div>
      <table class="ar-table">
        <tr><th>ID</th><th>Status</th><th>BPB</th><th>Category</th><th>Parent</th><th>Description</th></tr>
        {rows_html}
      </table>
    </div>

    <script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
    <script>
    (function() {{
      const ctx = document.getElementById('arChart');
      if (!ctx) return;
      new Chart(ctx, {{
        type: 'scatter',
        data: {{
          labels: {chart_labels},
          datasets: [{{
            label: 'Experiments',
            data: {chart_bpbs}.map((v, i) => ({{x: i, y: v}})),
            backgroundColor: {chart_colors},
            borderColor: {chart_borders},
            pointRadius: 6, borderWidth: 1.5
          }}, {{
            label: 'Running Best',
            data: {chart_running}.map((v, i) => ({{x: i, y: v}})),
            type: 'line', borderColor: '#27ae60', borderWidth: 2,
            pointRadius: 0, fill: false, stepped: 'after'
          }}]
        }},
        options: {{
          responsive: true,
          plugins: {{ legend: {{ display: true, position: 'top', labels: {{ font: {{ size: 11 }} }} }} }},
          scales: {{
            x: {{ title: {{ display: true, text: 'Experiment #' }}, ticks: {{ font: {{ size: 10 }} }} }},
            y: {{ title: {{ display: true, text: 'val_bpb (lower is better)' }}, ticks: {{ font: {{ size: 10 }} }} }}
          }}
        }}
      }});
    }})();
    </script>
    """
    return html

# ─── Dual-Model Agent ───────────────────────────────────────────────

BRAINSTORM_PROMPT = f"""You are an ML researcher brainstorming experiment ideas for a GPT training script.

Given the current code and past results, propose exactly 3 diverse experiment ideas.
Each idea should be a DIFFERENT type of change (architecture, optimizer, hyperparameter, etc.).

Categories: {', '.join(CATEGORIES)}

Respond with EXACTLY this JSON (no other text):
{{
  "ideas": [
    {{"description": "what to change and why", "category": "one of the categories", "risk": "low|medium|high", "expected_impact": "small|medium|large"}},
    {{"description": "...", "category": "...", "risk": "...", "expected_impact": "..."}},
    {{"description": "...", "category": "...", "risk": "...", "expected_impact": "..."}}
  ]
}}"""

IMPLEMENT_PROMPT = f"""You are an autonomous ML researcher implementing a specific experiment idea on a GPT training script.

RULES:
- You can ONLY modify train.py. Never change prepare.py.
- Training runs for a fixed 5-minute time budget.
- Goal: lower val_bpb.
- You CANNOT add new package imports. Only use what's available.
- The attention uses PyTorch SDPA, NOT Flash Attention 3.
- This runs on a Colab GPU with limited VRAM. Don't make the model too large.
- Keep changes focused. Implement exactly the requested idea.

Categories: {', '.join(CATEGORIES)}

Respond with EXACTLY this JSON (no other text):
{{
  "description": "short description of what you changed",
  "category": "one of: {', '.join(CATEGORIES)}",
  "train_py": "the complete modified train.py file content"
}}"""

def brainstorm_ideas(train_py, results_summary, prepare_py):
    """Use Haiku to quickly brainstorm 3 diverse ideas."""
    user_msg = f"""Current train.py:\n```python\n{train_py}\n```\n\nExperiment history:\n```\n{results_summary}\n```\n\nPropose 3 diverse experiment ideas."""

    response = client.messages.create(
        model=brainstorm_model,
        max_tokens=2000,
        system=BRAINSTORM_PROMPT,
        messages=[{"role": "user", "content": user_msg}]
    )
    text = response.content[0].text.strip()
    if text.startswith("```"): text = re.sub(r'^```(?:json)?\n?', '', text)
    if text.endswith("```"): text = text[:-3]
    return json.loads(text.strip())["ideas"]


def implement_idea(train_py, prepare_py, idea_desc, results_summary, error_context=None):
    """Use Sonnet to implement a specific idea."""
    user_msg = f"""## Idea to implement\n{idea_desc}\n\n## prepare.py (READ-ONLY)\n```python\n{prepare_py}\n```\n\n## Current train.py\n```python\n{train_py}\n```\n\n## Results so far\n```\n{results_summary}\n```"""
    if error_context:
        user_msg += f"""\n\n## Previous attempt FAILED:\n```\n{error_context}\n```\nFix the issue or try a different approach."""
    user_msg += "\n\nImplement this idea. Return the COMPLETE modified train.py as JSON."

    response = client.messages.create(
        model=implementation_model,
        max_tokens=16000,
        system=IMPLEMENT_PROMPT,
        messages=[{"role": "user", "content": user_msg}]
    )
    text = response.content[0].text.strip()
    if text.startswith("```"): text = re.sub(r'^```(?:json)?\n?', '', text)
    if text.endswith("```"): text = text[:-3]
    parsed = json.loads(text.strip())
    return parsed["description"], parsed.get("category", "other"), parsed["train_py"]


def single_model_proposal(train_py, prepare_py, results_summary, error_context=None):
    """Fallback: single model proposes + implements."""
    return implement_idea(train_py, prepare_py, "Propose and implement your best idea to lower val_bpb.", results_summary, error_context)


def build_results_summary(ckpt):
    """Build a compact text summary of experiments for the agent."""
    lines = ["id\tval_bpb\tstatus\tcategory\tparent\tdescription"]
    for e in ckpt["experiments"]:
        bpb = f"{e['val_bpb']:.6f}" if e['val_bpb'] > 0 else '0.000000'
        lines.append(f"{e['id']}\t{bpb}\t{e['status']}\t{e.get('category','')}\t{e.get('parent_id','')}\t{e['description']}")
    return "\n".join(lines)


# ─── Main Loop ──────────────────────────────────────────────────────

ckpt = load_checkpoint()
prepare_py = read_file("prepare.py")
start_from = ckpt["experiments_completed"]
dashboard_display = display(HTML(""), display_id=True)

if start_from > 0:
    print(f"RESUMING from experiment {start_from + 1} (auto-resume from checkpoint)")
    # Restore best train.py
    write_file("train.py", ckpt["best_train_py"])

print(f"Experiments: {start_from + 1} to {start_from + max_experiments}")
print(f"{'Dual-model' if use_dual_model else 'Single-model'}: ", end="")
if use_dual_model:
    print(f"{brainstorm_model} (brainstorm) + {implementation_model} (implement)")
else:
    print(f"{implementation_model}")
print(f"Baseline: {ckpt['baseline_bpb']:.6f} | Current best: {ckpt['best_bpb']:.6f}")
print("=" * 60)

error_context = None

for exp_idx in range(max_experiments):
    exp_num = start_from + exp_idx + 1
    exp_id = f"exp{exp_num:03d}"
    last_kept_id = [e["id"] for e in ckpt["experiments"] if e["status"] == "keep"][-1]

    # Update dashboard with "thinking" status
    dashboard_display.update(HTML(render_dashboard(ckpt, f"Experiment {exp_num}: Agent is thinking...")))

    current_train = read_file("train.py")
    results_summary = build_results_summary(ckpt)

    # ── Step 1: Get proposal ──
    description = category = new_train = None
    try:
        if use_dual_model and not error_context:
            # Dual-model: brainstorm then implement
            ideas = brainstorm_ideas(current_train, results_summary, prepare_py)
            # Pick the idea with best expected_impact:risk ratio
            risk_score = {"low": 1, "medium": 2, "high": 3}
            impact_score = {"small": 1, "medium": 2, "large": 3}
            scored = [(impact_score.get(i.get("expected_impact","medium"),2) / risk_score.get(i.get("risk","medium"),2), i) for i in ideas]
            scored.sort(key=lambda x: -x[0])
            best_idea = scored[0][1]
            print(f"  Brainstorm: {len(ideas)} ideas, picked: {best_idea['description'][:60]}")
            description, category, new_train = implement_idea(
                current_train, prepare_py, best_idea["description"], results_summary
            )
        else:
            # Single model or retrying after error
            description, category, new_train = single_model_proposal(
                current_train, prepare_py, results_summary, error_context
            )
        error_context = None
    except Exception as e:
        print(f"  Agent error: {e}")
        error_context = str(e)
        ckpt["experiments"].append({
            "id": exp_id, "parent_id": last_kept_id, "val_bpb": 0,
            "memory_gb": 0, "status": "crash", "category": "other",
            "description": f"Agent API error: {str(e)[:80]}",
            "timestamp": time.time(), "elapsed_sec": 0
        })
        ckpt["experiments_completed"] = exp_num
        save_checkpoint(ckpt)
        continue

    if category not in CATEGORIES:
        category = "other"

    # ── Step 2: Syntax pre-validation ──
    valid, syntax_err = validate_syntax(new_train)
    if not valid:
        print(f"  SYNTAX FAIL: {syntax_err[:120]}")
        ckpt["experiments"].append({
            "id": exp_id, "parent_id": last_kept_id, "val_bpb": 0,
            "memory_gb": 0, "status": "syntax_fail", "category": category,
            "description": f"[syntax error] {description[:80]}",
            "timestamp": time.time(), "elapsed_sec": 0
        })
        ckpt["experiments_completed"] = exp_num
        save_checkpoint(ckpt)
        error_context = f"Syntax error: {syntax_err}"
        dashboard_display.update(HTML(render_dashboard(ckpt, f"Exp {exp_num}: Syntax error caught, skipping training")))
        continue

    # ── Step 3: Run training ──
    dashboard_display.update(HTML(render_dashboard(ckpt, f"Exp {exp_num}: Training... ({description[:50]})")))
    write_file("train_backup.py", current_train)
    write_file("train.py", new_train)

    t0 = time.time()
    val_bpb, peak_vram, success, err = run_training()
    elapsed = time.time() - t0

    # ── Step 4: Evaluate ──
    if not success:
        status = "crash"
        print(f"  CRASH ({elapsed:.0f}s): {err[:150]}")
        write_file("train.py", current_train)
        error_context = err
        exp_record = {
            "id": exp_id, "parent_id": last_kept_id, "val_bpb": 0,
            "memory_gb": 0, "status": "crash", "category": category,
            "description": description[:100],
            "timestamp": time.time(), "elapsed_sec": elapsed
        }
    elif val_bpb < ckpt["best_bpb"]:
        improvement = ckpt["best_bpb"] - val_bpb
        print(f"  KEEP! bpb={val_bpb:.6f} (improved {improvement:.6f}) [{elapsed:.0f}s]")
        ckpt["best_bpb"] = val_bpb
        ckpt["best_train_py"] = new_train
        status = "keep"
        exp_record = {
            "id": exp_id, "parent_id": last_kept_id, "val_bpb": val_bpb,
            "memory_gb": round(peak_vram/1024, 1), "status": "keep", "category": category,
            "description": description[:100],
            "timestamp": time.time(), "elapsed_sec": elapsed
        }
    else:
        print(f"  DISCARD: bpb={val_bpb:.6f} (best={ckpt['best_bpb']:.6f}) [{elapsed:.0f}s]")
        write_file("train.py", current_train)
        status = "discard"
        exp_record = {
            "id": exp_id, "parent_id": last_kept_id, "val_bpb": val_bpb,
            "memory_gb": round(peak_vram/1024, 1), "status": "discard", "category": category,
            "description": description[:100],
            "timestamp": time.time(), "elapsed_sec": elapsed
        }

    ckpt["experiments"].append(exp_record)
    ckpt["experiments_completed"] = exp_num
    save_checkpoint(ckpt)

    # Update dashboard
    dashboard_display.update(HTML(render_dashboard(ckpt)))

# Final dashboard
improvement = ckpt['baseline_bpb'] - ckpt['best_bpb']
pct = improvement / ckpt['baseline_bpb'] * 100
dashboard_display.update(HTML(render_dashboard(ckpt,
    f"Complete! {max_experiments} experiments done. Improvement: {improvement:.6f} ({pct:.2f}%)"
)))
print(f"\nDone! Best val_bpb: {ckpt['best_bpb']:.6f} (improved {improvement:.6f} / {pct:.2f}%)")

---
## Cell 7: Generate HTML Summary Report

In [ ]:
#@title Cell 7: Generate Downloadable HTML Report
#@markdown Creates a beautiful, self-contained HTML report of all experiments.
#@markdown Inspired by The AI Scientist's automated paper generation.

import json, os
from datetime import datetime
from IPython.display import display, HTML

os.chdir("/content/autoresearch")
ckpt = json.load(open("checkpoint.json"))
exps = ckpt["experiments"]
baseline = ckpt["baseline_bpb"]
best = ckpt["best_bpb"]
improvement = baseline - best
pct = improvement / baseline * 100 if baseline > 0 else 0
n_total = len(exps) - 1
n_keep = sum(1 for e in exps if e["status"] == "keep") - 1
n_discard = sum(1 for e in exps if e["status"] == "discard")
n_crash = sum(1 for e in exps if e["status"] == "crash")
n_syntax = sum(1 for e in exps if e["status"] == "syntax_fail")
keep_rate = (n_keep / n_total * 100) if n_total > 0 else 0

# Diff: baseline vs best train.py
import difflib
baseline_lines = open("train_original.py").readlines()
best_lines = ckpt["best_train_py"].splitlines(keepends=True)
diff_html = ""
for line in difflib.unified_diff(baseline_lines, best_lines, fromfile="baseline train.py", tofile="best train.py", lineterm=""):
    line_esc = line.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    if line.startswith("+") and not line.startswith("+++"):
        diff_html += f'<div style="background:#e6ffec;padding:1px 8px;font-family:monospace;font-size:12px;white-space:pre-wrap">{line_esc}</div>'
    elif line.startswith("-") and not line.startswith("---"):
        diff_html += f'<div style="background:#ffebe9;padding:1px 8px;font-family:monospace;font-size:12px;white-space:pre-wrap">{line_esc}</div>'
    elif line.startswith("@@"):
        diff_html += f'<div style="background:#ddf4ff;padding:1px 8px;font-family:monospace;font-size:12px;color:#0969da;white-space:pre-wrap">{line_esc}</div>'
    else:
        diff_html += f'<div style="padding:1px 8px;font-family:monospace;font-size:12px;color:#666;white-space:pre-wrap">{line_esc}</div>'

# Category analysis
cat_stats = {}
for e in exps:
    c = e.get("category", "other")
    if c == "baseline": continue
    if c not in cat_stats: cat_stats[c] = {"total": 0, "kept": 0}
    cat_stats[c]["total"] += 1
    if e["status"] == "keep": cat_stats[c]["kept"] += 1

cat_table = ""
for cat in sorted(cat_stats.keys()):
    s = cat_stats[cat]
    rate = s["kept"] / s["total"] * 100 if s["total"] > 0 else 0
    bar_width = min(rate * 2, 100)
    bar_color = "#4caf50" if rate >= 40 else "#ff9800" if rate >= 15 else "#f44336"
    cat_table += f'<tr><td style="font-weight:600">{cat}</td><td>{s["total"]}</td><td>{s["kept"]}</td><td><div style="background:#eee;border-radius:4px;height:16px;width:120px"><div style="background:{bar_color};height:16px;width:{bar_width}%;border-radius:4px;text-align:center;color:white;font-size:10px;line-height:16px">{rate:.0f}%</div></div></td></tr>'

# Full experiment log
exp_rows = ""
for e in exps:
    s = e["status"]
    color = {"keep":"#e8f5e9", "discard":"#fff", "crash":"#ffebee", "syntax_fail":"#fff3e0"}.get(s, "#fff")
    icon = {"keep":"&#10004;", "discard":"&#10008;", "crash":"&#9888;", "syntax_fail":"&#9998;"}.get(s, "")
    bpb = f"{e['val_bpb']:.6f}" if e['val_bpb'] > 0 else "-"
    exp_rows += f'<tr style="background:{color}"><td>{e["id"]}</td><td>{icon} {s}</td><td style="font-family:monospace">{bpb}</td><td>{e.get("category","")}</td><td>{e.get("parent_id","-") or "-"}</td><td>{e["description"]}</td></tr>'

# Kept experiments detail
kept_details = ""
kept_exps = [e for e in exps if e["status"] == "keep"]
for i, e in enumerate(kept_exps):
    delta = ""
    if i > 0:
        d = kept_exps[i-1]["val_bpb"] - e["val_bpb"]
        delta = f' <span style="color:#2e7d32;font-weight:600">(&#9660; {d:.6f})</span>'
    kept_details += f'<div style="padding:8px 12px;margin:4px 0;background:#f1f8e9;border-radius:6px;border-left:4px solid #4caf50"><b>{e["id"]}</b> &mdash; {e["val_bpb"]:.6f}{delta}<br><span style="color:#555;font-size:13px">{e["description"]}</span></div>'

# Chart data
valid = [e for e in exps if e["status"] in ("keep", "discard")]
chart_labels = json.dumps([e["id"] for e in valid])
chart_bpbs = json.dumps([e["val_bpb"] for e in valid])
chart_colors = json.dumps(["#4caf50" if e["status"]=="keep" else "#ccc" for e in valid])
running = []
cur = baseline
for e in valid:
    if e["status"] == "keep" and e["val_bpb"] < cur: cur = e["val_bpb"]
    running.append(cur)
chart_running = json.dumps(running)

report_html = f"""
<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Autoresearch v2 Report</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 900px; margin: 40px auto; padding: 0 20px; color: #333; line-height: 1.6; }}
  h1 {{ border-bottom: 3px solid #4caf50; padding-bottom: 12px; }}
  h2 {{ color: #2e7d32; margin-top: 40px; }}
  .hero {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin: 24px 0; }}
  .hero-card {{ background: linear-gradient(135deg, #f8f9fa, #e8f5e9); border-radius: 12px; padding: 20px; text-align: center; border: 1px solid #e0e0e0; }}
  .hero-val {{ font-size: 32px; font-weight: 800; color: #2e7d32; }}
  .hero-lbl {{ font-size: 12px; color: #888; text-transform: uppercase; letter-spacing: 1px; margin-top: 4px; }}
  table {{ width: 100%; border-collapse: collapse; margin: 16px 0; font-size: 13px; }}
  th {{ background: #f5f5f5; padding: 10px; text-align: left; font-size: 11px; text-transform: uppercase; color: #666; border-bottom: 2px solid #e0e0e0; }}
  td {{ padding: 8px 10px; border-bottom: 1px solid #f0f0f0; }}
  .diff-container {{ max-height: 500px; overflow-y: auto; border: 1px solid #e0e0e0; border-radius: 8px; margin: 16px 0; }}
  .footer {{ margin-top: 40px; padding: 20px 0; border-top: 1px solid #e0e0e0; color: #999; font-size: 12px; }}
</style></head><body>

<h1>Autoresearch v2 &mdash; Experiment Report</h1>
<p style="color:#666">Generated {datetime.now().strftime('%Y-%m-%d %H:%M')} | GPU: {ckpt.get('gpu_tier','?')} | {n_total} experiments</p>

<div class="hero">
  <div class="hero-card"><div class="hero-val">{best:.4f}</div><div class="hero-lbl">Best BPB</div></div>
  <div class="hero-card"><div class="hero-val">{baseline:.4f}</div><div class="hero-lbl">Baseline</div></div>
  <div class="hero-card"><div class="hero-val" style="color:#e65100">{pct:.2f}%</div><div class="hero-lbl">Improvement</div></div>
  <div class="hero-card"><div class="hero-val">{keep_rate:.0f}%</div><div class="hero-lbl">Keep Rate</div></div>
</div>

<div class="hero" style="grid-template-columns: repeat(4, 1fr);">
  <div style="text-align:center"><span style="font-size:24px;font-weight:700;color:#4caf50">{n_keep}</span><br><span style="font-size:11px;color:#888">KEPT</span></div>
  <div style="text-align:center"><span style="font-size:24px;font-weight:700;color:#999">{n_discard}</span><br><span style="font-size:11px;color:#888">DISCARDED</span></div>
  <div style="text-align:center"><span style="font-size:24px;font-weight:700;color:#f44336">{n_crash}</span><br><span style="font-size:11px;color:#888">CRASHED</span></div>
  <div style="text-align:center"><span style="font-size:24px;font-weight:700;color:#ff9800">{n_syntax}</span><br><span style="font-size:11px;color:#888">SYNTAX FAIL</span></div>
</div>

<h2>Progress Chart</h2>
<canvas id="reportChart" height="120"></canvas>

<h2>Winning Experiments</h2>
{kept_details}

<h2>Category Analysis</h2>
<p style="font-size:13px;color:#666">Which types of changes were most productive?</p>
<table><tr><th>Category</th><th>Tried</th><th>Kept</th><th>Success Rate</th></tr>
{cat_table}
</table>

<h2>Best vs Baseline Diff</h2>
<div class="diff-container">{diff_html}</div>

<h2>Full Experiment Log</h2>
<table><tr><th>ID</th><th>Status</th><th>BPB</th><th>Category</th><th>Parent</th><th>Description</th></tr>
{exp_rows}
</table>

<div class="footer">
  Generated by Autoresearch v2 (Colab Enhanced Edition)<br>
  Based on <a href="https://github.com/karpathy/autoresearch">Karpathy's autoresearch</a>
  | Enhanced with ideas from AlphaEvolve, AIDE, pi-autoresearch, The AI Scientist
</div>

<script>
new Chart(document.getElementById('reportChart'), {{
  type: 'scatter',
  data: {{
    datasets: [{{
      label: 'Experiments',
      data: {chart_bpbs}.map((v, i) => ({{x: i, y: v}})),
      backgroundColor: {chart_colors},
      pointRadius: 7, borderWidth: 1.5, borderColor: '#999'
    }}, {{
      label: 'Running Best',
      data: {chart_running}.map((v, i) => ({{x: i, y: v}})),
      type: 'line', borderColor: '#4caf50', borderWidth: 2.5,
      pointRadius: 0, fill: false, stepped: 'after'
    }}]
  }},
  options: {{
    responsive: true,
    plugins: {{ legend: {{ position: 'top' }} }},
    scales: {{
      x: {{ title: {{ display: true, text: 'Experiment #' }} }},
      y: {{ title: {{ display: true, text: 'val_bpb (lower is better)' }} }}
    }}
  }}
}});
</script>
</body></html>
"""

# Save report
with open("report.html", "w") as f:
    f.write(report_html)

print("Report saved to report.html")
print("Preview:")
display(HTML(report_html))

---
## Cell 8: Download Everything

In [ ]:
#@title Cell 8: Download Results
#@markdown Downloads the report, checkpoint, and best train.py.

from google.colab import files
import os
os.chdir("/content/autoresearch")

for f in ["report.html", "checkpoint.json", "train.py", "train_original.py"]:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded {f}")